# Chapter 9.4: Benchmarking Methodology

Accurate benchmarking is critical for capacity planning, model selection, and hardware decisions.
This notebook covers systematic approaches to benchmarking LLM serving.

## Learning Objectives
- Use vLLM and SGLang built-in benchmark tools
- Build custom async benchmarking tools
- Generate realistic workloads (ShareGPT, synthetic Poisson)
- Collect and analyze metrics with statistical rigor
- Create publication-quality benchmark charts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part9/chapter_9.4_benchmarking.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part9/chapter_9.4_benchmarking.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import json
import time
import random
import math
import asyncio
import statistics
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict

print("Setup complete.")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_benchmark_methodology():
    """Benchmark methodology diagram: Define -> Configure -> Warm up -> Run -> Collect -> Analyze -> Report."""
    fig, ax = plt.subplots(figsize=(18, 5))
    ax.set_xlim(0, 20); ax.set_ylim(0, 5); ax.axis('off')
    ax.set_title('Benchmark Methodology Pipeline', fontsize=16, fontweight='bold', pad=15)

    steps = [
        ('Define\nWorkload', '#4A90D9', 'ShareGPT /\nSynthetic / Fixed'),
        ('Configure\nSystem', '#27AE60', 'Model, GPU,\nBatch size, Quant'),
        ('Warm Up', '#F39C12', 'JIT compile,\nCUDA graphs'),
        ('Run\nBenchmark', '#E74C3C', 'Send requests\nat target rate'),
        ('Collect\nMetrics', '#8E44AD', 'Latency, TTFT,\nthroughput'),
        ('Analyze', '#4A90D9', 'Percentiles,\nCIs, outliers'),
        ('Report', '#27AE60', 'Charts, tables,\nrecommendations'),
    ]

    box_w, box_h = 2.2, 2.0
    y_center = 2.5
    x_start = 0.3
    spacing = 2.7

    for i, (label, color, detail) in enumerate(steps):
        x = x_start + i * spacing
        rect = FancyBboxPatch((x, y_center - box_h/2), box_w, box_h,
                              boxstyle='round,pad=0.12', facecolor=color,
                              edgecolor='white', linewidth=2, alpha=0.85)
        ax.add_patch(rect)
        ax.text(x + box_w/2, y_center + 0.25, label, ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')
        ax.text(x + box_w/2, y_center - 0.5, detail, ha='center', va='center',
                fontsize=7, color='white', alpha=0.9)

        # Arrow
        if i < len(steps) - 1:
            ax.annotate('', xy=(x + box_w + 0.15, y_center),
                        xytext=(x + box_w + spacing - box_w - 0.15, y_center),
                        arrowprops=dict(arrowstyle='<-', color='#333', lw=2))

    # Repeat loop arrow from Report back to Define
    ax.annotate('', xy=(x_start + box_w/2, y_center + box_h/2 + 0.5),
                xytext=(x_start + 6 * spacing + box_w/2, y_center + box_h/2 + 0.5),
                arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.5,
                                connectionstyle='arc3,rad=0.3', linestyle='dashed'))
    ax.text(10, 4.5, 'iterate & improve', ha='center', fontsize=8, color='#E74C3C', style='italic')

    plt.tight_layout()
    plt.savefig('benchmark_methodology.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_benchmark_methodology()

## 1. Demo: Using vLLM's benchmark_serving.py

vLLM provides `benchmark_serving.py` for systematic performance testing.

In [ ]:
# vLLM benchmark_serving.py usage and options
# This cell documents the command-line interface

VLLM_BENCHMARK_COMMANDS = {
    "basic": """\
python -m vllm.entrypoints.openai.api_server \\
    --model meta-llama/Llama-3.1-8B-Instruct \\
    --port 8000

# In another terminal:
python benchmarks/benchmark_serving.py \\
    --backend vllm \\
    --base-url http://localhost:8000 \\
    --model meta-llama/Llama-3.1-8B-Instruct \\
    --dataset-name sharegpt \\
    --dataset-path ShareGPT_V3_unfiltered_cleaned_split.json \\
    --num-prompts 1000 \\
    --request-rate 10
""",
    "fixed_length": """\
# Fixed input/output length for controlled comparison
python benchmarks/benchmark_serving.py \\
    --backend vllm \\
    --base-url http://localhost:8000 \\
    --model meta-llama/Llama-3.1-8B-Instruct \\
    --dataset-name random \\
    --random-input-len 512 \\
    --random-output-len 256 \\
    --num-prompts 500 \\
    --request-rate inf
""",
    "sweep_rates": """\
# Sweep request rates to find saturation point
for RATE in 1 2 5 10 20 50 100; do
    echo "=== Rate: $RATE req/s ==="
    python benchmarks/benchmark_serving.py \\
        --backend vllm \\
        --base-url http://localhost:8000 \\
        --model meta-llama/Llama-3.1-8B-Instruct \\
        --dataset-name random \\
        --random-input-len 256 \\
        --random-output-len 128 \\
        --num-prompts 200 \\
        --request-rate $RATE \\
        --save-result \\
        --result-dir results/ \\
        --result-filename "rate_${RATE}.json"
done
""",
}

for name, cmd in VLLM_BENCHMARK_COMMANDS.items():
    print(f"\n{'='*60}")
    print(f"Benchmark: {name}")
    print(f"{'='*60}")
    print(cmd)

In [ ]:
# Key benchmark metrics and how to interpret them

BENCHMARK_METRICS = {
    "Request throughput (req/s)": {
        "description": "How many complete requests are served per second",
        "higher_better": True,
        "typical_range": "1-100 depending on model size and hardware",
    },
    "Token throughput (tok/s)": {
        "description": "Total output tokens generated per second across all requests",
        "higher_better": True,
        "typical_range": "500-5000 for 7-8B models on A100",
    },
    "Time to First Token (TTFT)": {
        "description": "Latency from request submission to first generated token",
        "higher_better": False,
        "typical_range": "50ms-5s depending on input length and queue",
    },
    "Inter-Token Latency (ITL)": {
        "description": "Average time between consecutive output tokens",
        "higher_better": False,
        "typical_range": "10-100ms per token",
    },
    "End-to-End Latency (E2E)": {
        "description": "Total time from request to complete response",
        "higher_better": False,
        "typical_range": "0.5s-60s depending on output length",
    },
}

print(f"{'Metric':<35} {'Better':>8} {'Typical Range'}")
print("=" * 80)
for name, info in BENCHMARK_METRICS.items():
    direction = "Higher" if info["higher_better"] else "Lower"
    print(f"{name:<35} {direction:>8} {info['typical_range']}")

## 2. Demo: Using SGLang's bench_serving.py

In [ ]:
# SGLang benchmark commands

SGLANG_BENCHMARK_COMMANDS = {
    "basic": """\
# Start SGLang server
python -m sglang.launch_server \\
    --model-path meta-llama/Llama-3.1-8B-Instruct \\
    --port 30000

# Run benchmark
python -m sglang.bench_serving \\
    --backend sglang \\
    --base-url http://localhost:30000 \\
    --model meta-llama/Llama-3.1-8B-Instruct \\
    --dataset-name random \\
    --random-input-len 512 \\
    --random-output-len 256 \\
    --num-prompts 500 \\
    --request-rate 10
""",
    "sharegpt": """\
python -m sglang.bench_serving \\
    --backend sglang \\
    --base-url http://localhost:30000 \\
    --model meta-llama/Llama-3.1-8B-Instruct \\
    --dataset-name sharegpt \\
    --dataset-path ShareGPT_V3_unfiltered_cleaned_split.json \\
    --num-prompts 1000 \\
    --request-rate 5 \\
    --output-file results/sglang_sharegpt.jsonl
""",
    "compare_with_vllm": """\
# Use SGLang's bench to compare against vLLM
python -m sglang.bench_serving \\
    --backend vllm \\
    --base-url http://localhost:8000 \\
    --model meta-llama/Llama-3.1-8B-Instruct \\
    --dataset-name random \\
    --random-input-len 256 \\
    --random-output-len 128 \\
    --num-prompts 500 \\
    --request-rate 10
""",
}

for name, cmd in SGLANG_BENCHMARK_COMMANDS.items():
    print(f"\n--- {name} ---")
    print(cmd)

## 3. Demo: Build a Custom Benchmarking Tool

A flexible async benchmarking tool that supports various arrival patterns
and collects detailed per-request metrics.

In [ ]:
@dataclass
class RequestResult:
    """Result of a single benchmark request."""
    request_id: int
    input_tokens: int
    output_tokens: int
    ttft_ms: float           # Time to first token
    e2e_latency_ms: float    # End-to-end latency
    itl_ms: float            # Inter-token latency (average)
    tokens_per_second: float
    success: bool
    error: str = ""
    submit_time: float = 0.0
    start_time: float = 0.0
    end_time: float = 0.0


@dataclass
class BenchmarkConfig:
    """Configuration for a benchmark run."""
    base_url: str = "http://localhost:8000"
    model: str = "default"
    num_requests: int = 100
    request_rate: float = 10.0  # requests per second, inf for max
    input_len: int = 256
    output_len: int = 128
    input_len_range: tuple = (128, 512)  # For variable workloads
    output_len_range: tuple = (64, 256)
    variable_length: bool = False
    temperature: float = 0.0
    stream: bool = True
    arrival_pattern: str = "poisson"  # "poisson", "constant", "burst"


class LLMBenchmark:
    """Async HTTP benchmark tool for LLM serving endpoints."""
    
    def __init__(self, config: BenchmarkConfig):
        self.config = config
        self.results: list[RequestResult] = []
    
    def _generate_inter_arrival_times(self) -> list[float]:
        """Generate inter-arrival times based on the configured pattern."""
        n = self.config.num_requests
        rate = self.config.request_rate
        
        if rate == float('inf'):
            return [0.0] * n
        
        rng = random.Random(42)
        
        if self.config.arrival_pattern == "constant":
            return [1.0 / rate] * n
        elif self.config.arrival_pattern == "poisson":
            return [rng.expovariate(rate) for _ in range(n)]
        elif self.config.arrival_pattern == "burst":
            # Bursts of 10 requests, then pause
            times = []
            for i in range(n):
                if i % 10 < 9:
                    times.append(0.01)  # Quick burst
                else:
                    times.append(5.0 / rate)  # Pause between bursts
            return times
        else:
            raise ValueError(f"Unknown arrival pattern: {self.config.arrival_pattern}")
    
    def _generate_request(self, request_id: int) -> dict:
        """Generate a request payload."""
        rng = random.Random(request_id)
        
        if self.config.variable_length:
            input_len = rng.randint(*self.config.input_len_range)
            output_len = rng.randint(*self.config.output_len_range)
        else:
            input_len = self.config.input_len
            output_len = self.config.output_len
        
        # Generate dummy prompt of appropriate length
        prompt = "Summarize: " + " ".join(["word"] * (input_len // 2))
        
        return {
            "model": self.config.model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": output_len,
            "temperature": self.config.temperature,
            "stream": self.config.stream,
            "_meta": {"request_id": request_id, "input_len": input_len, "output_len": output_len},
        }
    
    async def _send_request(self, request_id: int, payload: dict) -> RequestResult:
        """Send a single request and collect metrics."""
        import urllib.request
        meta = payload.pop("_meta")
        submit_time = time.time()
        
        try:
            data = json.dumps(payload).encode()
            req = urllib.request.Request(
                f"{self.config.base_url}/v1/chat/completions",
                data=data,
                headers={"Content-Type": "application/json"}
            )
            
            start_time = time.time()
            with urllib.request.urlopen(req, timeout=120) as resp:
                first_byte_time = time.time()
                response_data = json.loads(resp.read())
                end_time = time.time()
            
            output_tokens = response_data.get("usage", {}).get("completion_tokens", meta["output_len"])
            e2e = (end_time - start_time) * 1000
            ttft = (first_byte_time - start_time) * 1000
            
            return RequestResult(
                request_id=request_id,
                input_tokens=meta["input_len"],
                output_tokens=output_tokens,
                ttft_ms=ttft,
                e2e_latency_ms=e2e,
                itl_ms=(e2e - ttft) / max(1, output_tokens - 1),
                tokens_per_second=output_tokens / (e2e / 1000) if e2e > 0 else 0,
                success=True,
                submit_time=submit_time,
                start_time=start_time,
                end_time=end_time,
            )
        except Exception as e:
            return RequestResult(
                request_id=request_id,
                input_tokens=meta["input_len"],
                output_tokens=0,
                ttft_ms=0, e2e_latency_ms=0, itl_ms=0, tokens_per_second=0,
                success=False, error=str(e),
                submit_time=submit_time,
            )
    
    def analyze_results(self) -> dict:
        """Compute summary statistics from benchmark results."""
        successful = [r for r in self.results if r.success]
        if not successful:
            return {"error": "No successful requests"}
        
        def percentile(values, p):
            s = sorted(values)
            idx = int(len(s) * p)
            return s[min(idx, len(s)-1)]
        
        e2e_latencies = [r.e2e_latency_ms for r in successful]
        ttfts = [r.ttft_ms for r in successful]
        itls = [r.itl_ms for r in successful if r.itl_ms > 0]
        tps_values = [r.tokens_per_second for r in successful if r.tokens_per_second > 0]
        
        total_time = max(r.end_time for r in successful) - min(r.submit_time for r in successful)
        total_output_tokens = sum(r.output_tokens for r in successful)
        
        return {
            "num_requests": len(self.results),
            "successful": len(successful),
            "failed": len(self.results) - len(successful),
            "total_duration_s": round(total_time, 2),
            "request_throughput": round(len(successful) / total_time, 2) if total_time > 0 else 0,
            "token_throughput": round(total_output_tokens / total_time, 2) if total_time > 0 else 0,
            "e2e_latency_ms": {
                "avg": round(statistics.mean(e2e_latencies), 1),
                "p50": round(percentile(e2e_latencies, 0.5), 1),
                "p95": round(percentile(e2e_latencies, 0.95), 1),
                "p99": round(percentile(e2e_latencies, 0.99), 1),
                "max": round(max(e2e_latencies), 1),
            },
            "ttft_ms": {
                "avg": round(statistics.mean(ttfts), 1),
                "p50": round(percentile(ttfts, 0.5), 1),
                "p95": round(percentile(ttfts, 0.95), 1),
                "p99": round(percentile(ttfts, 0.99), 1),
            },
            "itl_ms": {
                "avg": round(statistics.mean(itls), 1) if itls else 0,
                "p50": round(percentile(itls, 0.5), 1) if itls else 0,
                "p99": round(percentile(itls, 0.99), 1) if itls else 0,
            },
        }

print("LLMBenchmark class defined.")
print("Usage:")
print("  config = BenchmarkConfig(base_url='http://localhost:8000', num_requests=100)")
print("  bench = LLMBenchmark(config)")
print("  # Run the benchmark (requires a running server)")
print("  results = bench.analyze_results()")

## 4. Workload Generation: ShareGPT and Synthetic

In [ ]:
class WorkloadGenerator:
    """Generate realistic workloads for LLM benchmarking."""
    
    @staticmethod
    def sharegpt_distribution(n: int, seed: int = 42) -> list[tuple[int, int]]:
        """Generate input/output token pairs following ShareGPT distribution.
        
        ShareGPT characteristics:
        - Input lengths: log-normal, median ~200 tokens, long tail to 4000+
        - Output lengths: log-normal, median ~150 tokens, long tail to 2000+
        - Moderate correlation between input and output length
        """
        rng = random.Random(seed)
        pairs = []
        for _ in range(n):
            # Log-normal distribution for token lengths
            input_len = int(min(4096, max(10, rng.lognormvariate(5.3, 0.9))))
            # Output somewhat correlated with input
            output_base = rng.lognormvariate(4.8, 1.0)
            output_len = int(min(2048, max(5, output_base * (1 + 0.2 * input_len / 200))))
            pairs.append((input_len, output_len))
        return pairs
    
    @staticmethod
    def fixed_length(n: int, input_len: int, output_len: int) -> list[tuple[int, int]]:
        """Fixed-length workload for controlled experiments."""
        return [(input_len, output_len)] * n
    
    @staticmethod
    def code_generation(n: int, seed: int = 42) -> list[tuple[int, int]]:
        """Simulate code generation workload: short input, long output."""
        rng = random.Random(seed)
        return [
            (rng.randint(50, 300), rng.randint(200, 1500))
            for _ in range(n)
        ]
    
    @staticmethod
    def summarization(n: int, seed: int = 42) -> list[tuple[int, int]]:
        """Simulate summarization workload: long input, short output."""
        rng = random.Random(seed)
        return [
            (rng.randint(1000, 4000), rng.randint(50, 200))
            for _ in range(n)
        ]
    
    @staticmethod
    def poisson_arrivals(rate: float, duration: float, seed: int = 42) -> list[float]:
        """Generate Poisson arrival times."""
        rng = random.Random(seed)
        times = []
        t = 0
        while t < duration:
            t += rng.expovariate(rate)
            if t < duration:
                times.append(t)
        return times


# Demonstrate workload characteristics
wg = WorkloadGenerator()

workloads = {
    "ShareGPT": wg.sharegpt_distribution(1000),
    "Code Gen": wg.code_generation(1000),
    "Summarization": wg.summarization(1000),
    "Fixed (256/128)": wg.fixed_length(1000, 256, 128),
}

print(f"{'Workload':<20} {'Avg Input':>10} {'Avg Output':>11} {'Med Input':>10} "
      f"{'Med Output':>11} {'Max Input':>10} {'Max Output':>11}")
print("=" * 95)
for name, pairs in workloads.items():
    inputs = [p[0] for p in pairs]
    outputs = [p[1] for p in pairs]
    print(f"{name:<20} {statistics.mean(inputs):>10.0f} {statistics.mean(outputs):>11.0f} "
          f"{statistics.median(inputs):>10.0f} {statistics.median(outputs):>11.0f} "
          f"{max(inputs):>10} {max(outputs):>11}")

In [ ]:
# Visualize workload distributions using text-based histograms

def text_histogram(values, bins=20, width=50, title=""):
    """Print a text-based histogram."""
    min_val = min(values)
    max_val = max(values)
    bin_width = (max_val - min_val) / bins
    
    counts = [0] * bins
    for v in values:
        idx = min(bins - 1, int((v - min_val) / bin_width))
        counts[idx] += 1
    
    max_count = max(counts)
    
    print(f"\n{title}")
    print(f"  n={len(values)}, min={min_val:.0f}, max={max_val:.0f}, "
          f"mean={statistics.mean(values):.0f}, median={statistics.median(values):.0f}")
    print()
    
    for i in range(bins):
        lo = min_val + i * bin_width
        hi = lo + bin_width
        bar_len = int(counts[i] / max_count * width) if max_count > 0 else 0
        bar = "#" * bar_len
        print(f"  {lo:>7.0f}-{hi:<7.0f} | {bar:<{width}} {counts[i]}")

# Show ShareGPT distribution
sharegpt = wg.sharegpt_distribution(5000)
text_histogram([p[0] for p in sharegpt], bins=15, title="ShareGPT Input Token Distribution")
text_histogram([p[1] for p in sharegpt], bins=15, title="ShareGPT Output Token Distribution")

## 5. Metrics Collection and Statistical Analysis

In [ ]:
# Simulate benchmark results for analysis demonstration

def simulate_benchmark_results(config_name: str, n: int = 500, seed: int = 42) -> list[RequestResult]:
    """Generate simulated benchmark results for demo purposes."""
    rng = random.Random(seed)
    results = []
    
    # Different profiles for different configs
    profiles = {
        "vllm_a100": {"ttft_base": 80, "itl_base": 15, "throughput": 2500},
        "vllm_l40": {"ttft_base": 120, "itl_base": 22, "throughput": 1800},
        "sglang_a100": {"ttft_base": 70, "itl_base": 13, "throughput": 2800},
        "vllm_quantized": {"ttft_base": 60, "itl_base": 12, "throughput": 3200},
    }
    profile = profiles.get(config_name, profiles["vllm_a100"])
    
    base_time = time.time()
    
    for i in range(n):
        input_len = int(rng.lognormvariate(5.3, 0.8))
        output_len = int(rng.lognormvariate(4.8, 0.9))
        input_len = max(10, min(4096, input_len))
        output_len = max(5, min(2048, output_len))
        
        # TTFT scales with input length and queue
        ttft = profile["ttft_base"] * (1 + input_len / 1000) * rng.uniform(0.5, 2.0)
        # ITL with some variance
        itl = profile["itl_base"] * rng.uniform(0.7, 1.5)
        # E2E = TTFT + output_len * ITL
        e2e = ttft + output_len * itl
        
        submit_time = base_time + i * 0.1
        
        results.append(RequestResult(
            request_id=i,
            input_tokens=input_len,
            output_tokens=output_len,
            ttft_ms=ttft,
            e2e_latency_ms=e2e,
            itl_ms=itl,
            tokens_per_second=output_len / (e2e / 1000) if e2e > 0 else 0,
            success=rng.random() > 0.01,  # 1% failure rate
            submit_time=submit_time,
            start_time=submit_time + 0.01,
            end_time=submit_time + e2e / 1000,
        ))
    
    return results


# Generate and analyze results
configs = ["vllm_a100", "sglang_a100", "vllm_l40", "vllm_quantized"]
all_results = {}

for config in configs:
    results = simulate_benchmark_results(config, n=500)
    bench = LLMBenchmark(BenchmarkConfig())
    bench.results = results
    analysis = bench.analyze_results()
    all_results[config] = analysis

# Print comparison table
print(f"{'Config':<18} {'Req/s':>7} {'Tok/s':>8} {'E2E P50':>8} {'E2E P99':>8} "
      f"{'TTFT P50':>9} {'TTFT P99':>9} {'ITL P50':>8}")
print("=" * 95)
for config, a in all_results.items():
    print(f"{config:<18} {a['request_throughput']:>7.1f} {a['token_throughput']:>8.0f} "
          f"{a['e2e_latency_ms']['p50']:>7.0f}ms {a['e2e_latency_ms']['p99']:>7.0f}ms "
          f"{a['ttft_ms']['p50']:>8.0f}ms {a['ttft_ms']['p99']:>8.0f}ms "
          f"{a['itl_ms']['p50']:>7.1f}ms")

## 6. Comparing Results Fairly

In [ ]:
# Benchmark comparison checklist

COMPARISON_CHECKLIST = {
    "Controlled Variables": [
        "Same model and model weights (exact same checkpoint)",
        "Same quantization (FP16 vs AWQ vs GPTQ vs FP8)",
        "Same hardware (GPU model, count, memory)",
        "Same input/output token distribution",
        "Same request rate and arrival pattern",
        "Same max_model_len and context length",
        "Same number of requests (sufficient for statistical significance)",
    ],
    "Warm-up": [
        "Run warm-up requests before measuring (at least 50)",
        "Discard first 10% of results",
        "Ensure CUDA kernels are compiled and cached",
        "Ensure model weights are fully loaded to GPU",
    ],
    "Statistical Rigor": [
        "Run at least 3 repetitions",
        "Report confidence intervals, not just means",
        "Report percentiles (P50, P95, P99), not just averages",
        "Check for outliers and understand their cause",
        "Use sufficient sample size (>200 requests)",
    ],
    "Common Pitfalls": [
        "DO NOT compare different request rates as throughput",
        "DO NOT ignore the tail latency (P99+)",
        "DO NOT benchmark with temperature=0 only (affects batching)",
        "DO NOT compare at idle load only (test at saturation too)",
        "DO NOT forget to report the exact vLLM/SGLang version",
    ],
}

for category, items in COMPARISON_CHECKLIST.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  [ ] {item}")

---
## Hands-on Assignments

### Assignment 1: Benchmark vLLM with Different Batch Sizes

In [ ]:
# ASSIGNMENT 1: Analyze how max_num_seqs (max batch size) affects performance

def simulate_batch_size_sweep(
    batch_sizes: list[int] = [1, 4, 8, 16, 32, 64, 128, 256],
    num_requests: int = 500,
    input_len: int = 256,
    output_len: int = 128,
) -> dict:
    """Simulate benchmarks at different batch sizes.
    
    TODO: For each batch size, simulate the throughput and latency tradeoff.
    
    The relationship should model:
    - Throughput increases with batch size (more parallelism)
    - But with diminishing returns (memory bandwidth limited)
    - Latency increases with batch size (more contention)
    - TTFT increases more sharply (prefill competes with decode)
    
    Use these formulas:
    - throughput = base_throughput * min(batch_size, optimal_batch) / optimal_batch * 
                   (1 - 0.1 * max(0, batch_size - optimal_batch) / optimal_batch)
    - latency = base_latency * (1 + 0.5 * log2(batch_size))
    - ttft = base_ttft * (1 + 0.8 * log2(batch_size))
    
    Returns:
        dict mapping batch_size to metrics dict
    """
    # TODO: Implement the simulation
    results = {}
    
    base_throughput = 100  # tok/s at batch=1
    base_latency = 200    # ms at batch=1
    base_ttft = 50        # ms at batch=1
    optimal_batch = 64    # Sweet spot
    
    for bs in batch_sizes:
        # YOUR CODE HERE
        # Calculate throughput, latency, and ttft for this batch size
        pass
    
    return results


# Print results when implemented
results = simulate_batch_size_sweep()
if results:
    print(f"{'Batch':>6} {'Throughput':>12} {'E2E P50':>10} {'TTFT P50':>10} {'Efficiency':>12}")
    print("-" * 55)
    for bs, metrics in sorted(results.items()):
        print(f"{bs:>6} {metrics.get('throughput', 0):>10.0f} tok/s "
              f"{metrics.get('latency', 0):>9.0f}ms {metrics.get('ttft', 0):>9.0f}ms "
              f"{metrics.get('efficiency', 0):>10.1%}")
else:
    print("Implement the simulation to see results!")

In [ ]:
# ASSIGNMENT 1 - SOLUTION

def simulate_batch_size_sweep_solution(
    batch_sizes=[1, 4, 8, 16, 32, 64, 128, 256],
    num_requests=500,
):
    results = {}
    
    base_throughput = 100
    base_latency = 200
    base_ttft = 50
    optimal_batch = 64
    
    for bs in batch_sizes:
        # Throughput: scales up to optimal, then diminishes
        scale = min(bs, optimal_batch) / 1  # Linear scaling up to optimal
        penalty = max(0, (bs - optimal_batch) / optimal_batch) * 0.1
        throughput = base_throughput * scale * (1 - penalty)
        
        # Latency: increases logarithmically
        latency = base_latency * (1 + 0.5 * math.log2(max(1, bs)))
        
        # TTFT: increases more with batch size
        ttft_val = base_ttft * (1 + 0.8 * math.log2(max(1, bs)))
        
        # Efficiency: throughput per batch size (how well are we utilizing)
        efficiency = throughput / (base_throughput * bs) if bs > 0 else 0
        
        results[bs] = {
            "throughput": throughput,
            "latency": latency,
            "ttft": ttft_val,
            "efficiency": efficiency,
        }
    
    return results

results = simulate_batch_size_sweep_solution()
print(f"{'Batch':>6} {'Throughput':>12} {'E2E Latency':>12} {'TTFT':>10} {'Efficiency':>12}")
print("=" * 55)
for bs, m in sorted(results.items()):
    print(f"{bs:>6} {m['throughput']:>10.0f} tok/s {m['latency']:>10.0f}ms "
          f"{m['ttft']:>9.0f}ms {m['efficiency']:>10.1%}")

# Find optimal batch size
best_bs = max(results, key=lambda bs: results[bs]['throughput'])
print(f"\nOptimal batch size for throughput: {best_bs} "
      f"({results[best_bs]['throughput']:.0f} tok/s)")

### Assignment 2: Create a Realistic Workload Simulator

In [ ]:
# ASSIGNMENT 2: Build a workload simulator with diurnal patterns

class DiurnalWorkloadSimulator:
    """Simulate realistic daily traffic patterns for LLM serving.
    
    TODO: Implement a workload that varies over a 24-hour period:
    - Low traffic: 2 AM - 6 AM (20% of peak)
    - Ramp up: 6 AM - 9 AM
    - Peak: 9 AM - 12 PM and 2 PM - 5 PM (100%)
    - Lunch dip: 12 PM - 2 PM (60%)
    - Ramp down: 5 PM - 10 PM
    - Low: 10 PM - 2 AM (30%)
    
    The simulator should generate:
    - Request arrival times following time-varying Poisson process
    - Request types: 50% chat, 30% code, 20% summarization
    - Each type has different input/output distributions
    """
    
    def __init__(self, peak_rate: float = 50.0, seed: int = 42):
        self.peak_rate = peak_rate
        self.rng = random.Random(seed)
    
    def rate_at_hour(self, hour: float) -> float:
        """Get request rate at a given hour (0-24).
        TODO: Implement diurnal pattern.
        """
        # YOUR CODE HERE
        return self.peak_rate
    
    def generate_request(self) -> dict:
        """Generate a single request with type-appropriate parameters.
        TODO: Implement request generation.
        """
        # YOUR CODE HERE
        return {}
    
    def generate_day(self, start_hour: float = 0, end_hour: float = 24,
                     resolution_minutes: float = 1) -> list[dict]:
        """Generate a full day of traffic.
        TODO: Generate requests with correct arrival times.
        """
        # YOUR CODE HERE
        return []

sim = DiurnalWorkloadSimulator(peak_rate=50)
print("Implement the DiurnalWorkloadSimulator methods!")
print(f"Rate at 10 AM: {sim.rate_at_hour(10):.1f} req/s")
print(f"Rate at 3 AM: {sim.rate_at_hour(3):.1f} req/s")

In [ ]:
# ASSIGNMENT 2 - SOLUTION

class DiurnalWorkloadSimulatorSolution:
    def __init__(self, peak_rate=50.0, seed=42):
        self.peak_rate = peak_rate
        self.rng = random.Random(seed)
    
    def rate_at_hour(self, hour):
        h = hour % 24
        if 2 <= h < 6:
            return self.peak_rate * 0.2
        elif 6 <= h < 9:
            progress = (h - 6) / 3
            return self.peak_rate * (0.2 + 0.8 * progress)
        elif 9 <= h < 12:
            return self.peak_rate * 1.0
        elif 12 <= h < 14:
            return self.peak_rate * 0.6
        elif 14 <= h < 17:
            return self.peak_rate * 1.0
        elif 17 <= h < 22:
            progress = (h - 17) / 5
            return self.peak_rate * (1.0 - 0.7 * progress)
        else:  # 22-2
            return self.peak_rate * 0.3
    
    def generate_request(self):
        r = self.rng.random()
        if r < 0.5:  # Chat
            return {
                "type": "chat",
                "input_tokens": self.rng.randint(50, 500),
                "output_tokens": self.rng.randint(50, 300),
            }
        elif r < 0.8:  # Code
            return {
                "type": "code",
                "input_tokens": self.rng.randint(100, 400),
                "output_tokens": self.rng.randint(200, 1000),
            }
        else:  # Summarization
            return {
                "type": "summarization",
                "input_tokens": self.rng.randint(1000, 4000),
                "output_tokens": self.rng.randint(50, 200),
            }
    
    def generate_day(self, start_hour=0, end_hour=24, resolution_minutes=1):
        requests = []
        current_time = start_hour * 3600  # in seconds
        end_time = end_hour * 3600
        
        while current_time < end_time:
            hour = (current_time / 3600) % 24
            rate = self.rate_at_hour(hour)
            
            if rate > 0:
                inter_arrival = self.rng.expovariate(rate)
            else:
                inter_arrival = 60
            
            current_time += inter_arrival
            if current_time >= end_time:
                break
            
            req = self.generate_request()
            req["arrival_time"] = current_time
            req["hour"] = (current_time / 3600) % 24
            requests.append(req)
        
        return requests


sim = DiurnalWorkloadSimulatorSolution(peak_rate=50)

# Show rate throughout the day
print("Hourly request rate:")
for h in range(24):
    rate = sim.rate_at_hour(h)
    bar = "#" * int(rate)
    print(f"  {h:>2}:00  {rate:>5.1f} req/s  {bar}")

# Generate a day
day_requests = sim.generate_day()
print(f"\nTotal requests in 24h: {len(day_requests)}")

# Breakdown by type
type_counts = defaultdict(int)
for r in day_requests:
    type_counts[r['type']] += 1
print(f"By type: {dict(type_counts)}")

# Requests per hour
hourly = defaultdict(int)
for r in day_requests:
    hourly[int(r['hour'])] += 1
print(f"\nRequests per hour:")
for h in sorted(hourly):
    print(f"  {h:>2}:00  {hourly[h]:>5} requests")

### Assignment 3: Generate Publication-Quality Benchmark Charts

In [ ]:
# ASSIGNMENT 3: Create comprehensive benchmark comparison charts (text-based)

def create_benchmark_report(results: dict[str, dict], title: str = "Benchmark Report") -> str:
    """Create a comprehensive text-based benchmark report.
    
    TODO: Generate a report that includes:
    1. Summary table with all key metrics
    2. Throughput comparison bar chart (text-based)
    3. Latency percentile comparison
    4. Throughput vs latency tradeoff analysis
    5. Recommendation based on use case (latency-sensitive vs throughput)
    
    Args:
        results: dict mapping config_name to analysis dict (from analyze_results)
        title: Report title
    
    Returns:
        Formatted report string
    """
    # YOUR CODE HERE
    lines = [f"\n{'='*70}", f"{title:^70}", f"{'='*70}\n"]
    
    # TODO: Add summary table
    # TODO: Add text-based bar charts
    # TODO: Add recommendations
    
    return "\n".join(lines)

print("Implement create_benchmark_report to generate a full report!")

In [ ]:
# ASSIGNMENT 3 - SOLUTION

def create_benchmark_report_solution(results: dict, title="Benchmark Report"):
    lines = []
    lines.append(f"\n{'='*75}")
    lines.append(f"{title:^75}")
    lines.append(f"{'='*75}\n")
    
    # 1. Summary table
    lines.append("1. Summary Table")
    lines.append("-" * 75)
    header = (f"{'Config':<18} {'Req/s':>7} {'Tok/s':>8} {'E2E P50':>8} "
              f"{'E2E P99':>8} {'TTFT P50':>9} {'TTFT P99':>9}")
    lines.append(header)
    lines.append("-" * 75)
    
    for config, a in results.items():
        lines.append(
            f"{config:<18} {a['request_throughput']:>7.1f} {a['token_throughput']:>8.0f} "
            f"{a['e2e_latency_ms']['p50']:>7.0f}ms {a['e2e_latency_ms']['p99']:>7.0f}ms "
            f"{a['ttft_ms']['p50']:>8.0f}ms {a['ttft_ms']['p99']:>8.0f}ms"
        )
    
    # 2. Throughput bar chart
    lines.append(f"\n2. Token Throughput Comparison")
    lines.append("-" * 60)
    max_tp = max(a['token_throughput'] for a in results.values())
    for config, a in sorted(results.items(), key=lambda x: x[1]['token_throughput'], reverse=True):
        tp = a['token_throughput']
        bar = "#" * int(40 * tp / max_tp)
        lines.append(f"  {config:<18} {bar:<40} {tp:.0f} tok/s")
    
    # 3. Latency bar chart
    lines.append(f"\n3. E2E Latency P99 (lower is better)")
    lines.append("-" * 60)
    max_lat = max(a['e2e_latency_ms']['p99'] for a in results.values())
    for config, a in sorted(results.items(), key=lambda x: x[1]['e2e_latency_ms']['p99']):
        lat = a['e2e_latency_ms']['p99']
        bar = "#" * int(40 * lat / max_lat)
        lines.append(f"  {config:<18} {bar:<40} {lat:.0f}ms")
    
    # 4. Throughput-latency analysis
    lines.append(f"\n4. Throughput vs Latency Tradeoff")
    lines.append("-" * 60)
    for config, a in results.items():
        efficiency = a['token_throughput'] / max(1, a['e2e_latency_ms']['p50'])
        lines.append(f"  {config:<18} Efficiency: {efficiency:.2f} tok/s per ms latency")
    
    # 5. Recommendations
    lines.append(f"\n5. Recommendations")
    lines.append("-" * 60)
    
    best_throughput = max(results, key=lambda c: results[c]['token_throughput'])
    best_latency = min(results, key=lambda c: results[c]['e2e_latency_ms']['p99'])
    best_ttft = min(results, key=lambda c: results[c]['ttft_ms']['p99'])
    
    lines.append(f"  Throughput-optimized: {best_throughput} ({results[best_throughput]['token_throughput']:.0f} tok/s)")
    lines.append(f"  Latency-optimized:   {best_latency} (P99={results[best_latency]['e2e_latency_ms']['p99']:.0f}ms)")
    lines.append(f"  Interactive (TTFT):   {best_ttft} (P99={results[best_ttft]['ttft_ms']['p99']:.0f}ms)")
    
    lines.append(f"\n{'='*75}")
    return "\n".join(lines)


# Generate report from simulated data
report = create_benchmark_report_solution(all_results, "LLM Serving Benchmark: vLLM vs SGLang")
print(report)

In [ ]:
# Save report to file
report_path = Path("./benchmark_report.txt")
report_path.write_text(report)
print(f"Report saved to: {report_path.resolve()}")

## Summary

Key takeaways:

1. Use **built-in benchmark tools** (`benchmark_serving.py`, `bench_serving.py`) as your baseline
2. **ShareGPT workloads** better represent real traffic than fixed-length requests
3. **Poisson arrivals** are more realistic than constant-rate arrivals
4. Always report **percentiles** (P50, P95, P99), not just averages
5. **Control variables** carefully when comparing systems
6. **Warm up** before measuring and run multiple repetitions
7. The **throughput-latency tradeoff** is the fundamental tension in LLM serving